In [0]:
%run ../delta_function

In [0]:
import os
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.functions import current_timestamp
from pyspark.sql.types import TimestampType
import pyspark.sql.utils;
from pyspark.sql.types import StructType, StringType;
from pyspark.sql.functions import concat, lit, col, upper, max, when, row_number, to_date
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from datetime import datetime, timedelta
from pyspark.sql.functions import regexp_replace
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_date, datediff, when
spark.conf.set("spark.sql.execution.arrow.enabled", "true")

In [0]:
try:
    verbose_mode = dbutils.widgets.get("verbose_mode");
except:
    verbose_mode = 'debug'
try:
    current_division = dbutils.widgets.get("division");
except:
    current_division = 'mal'
try:
    current_environment = dbutils.widgets.get("environment");
except:
    current_environment = 'dev'
try:
    execution_mode = dbutils.widgets.get("execution_mode");
except:
    execution_mode = 'update'
try:
    current_project = dbutils.widgets.get("project");
except:
    current_project = 'maite_bi'
try:
    current_production_line = dbutils.widgets.get("production_line");
except:
    current_production_line = 'recommendations'

current_catalog = current_division + '_' + current_project + '_' + current_environment;

current_schema = current_production_line;

current_location = 'abfss://' + current_project + '@adlsdpcom'+ current_environment +f'data.dfs.core.windows.net/' + current_production_line + '/'

if verbose_mode == 'debug':
    display("Debug Mode")
    display(f"current_division : {current_division}")
    display(f"current_environment : {current_environment}")
    display(f"current_project : {current_project}")
    display(f"current_production_line : {current_production_line}")
    display(f"current_catalog : {current_catalog}")
    display(f"current_schema : {current_schema}")
    display(f"current_location : {current_location}")

In [0]:
if current_environment =='preprd':
    source = f"""ext_mal_psql_maite_vision_board_test.public"""    
else:
    source = f"""ext_mal_psql_maite_vision_board_{current_environment}.public"""

In [0]:
batches = f"""
SELECT id_batch, batch_number, b.production_line AS id_plant, ppl.name AS plant, requirement_specifications AS id_cdc, b.variety AS id_variety, b.planned_date, b.mes_number, pbst.label AS batch_status, harvest
FROM {source}.batches b
JOIN {source}.plants_production_lines ppl ON ppl.id_plant_production_line = b.production_line
JOIN {source}.parameters_batch_status_translations pbst ON pbst.status = b.status AND pbst.language = 1
WHERE b.deleted = false
"""

df_batches = spark.sql(batches)
df_batches.createOrReplaceTempView("batches")

In [0]:
reco_opti = f"""
SELECT batch, COUNT(batch) AS nb_reco_opti
FROM {source}.batches_production_planning bpp
WHERE bpp.deleted = false
GROUP BY batch
"""

df_reco_opti = spark.sql(reco_opti)
df_reco_opti.createOrReplaceTempView("reco_opti")

In [0]:
reco = f"""
SELECT r.id_recommendation, r.batch, r.calculation_date, r.status, r.acceptance_user, r.acceptance_comments, r.target_localization, r.calculation_localization, r.acceptance_status, b.plant, plt.label AS localization
FROM {source}.recommendations r
JOIN batches b ON b.id_batch = r.batch
JOIN {source}.parameters_localizations_translations plt ON plt.id_parameter_localization = r.target_localization
WHERE plt.language = 1 AND r.deleted = false
"""

df_reco = spark.sql(reco)

# Supprimer les lignes par condition
df_reco = df_reco.filter(~((F.col("plant") == "ROUEN1") & (F.col("localization") == "Germination J5")))
#df_reco = df_reco.filter(~((F.col("plant") == "NOGENT2") & (F.col("localization") == "Trempe J2")))        Retiré le 18/09/2024
#df_reco = df_reco.filter(~((F.col("plant") == "POLISY1") & (F.col("localization") == "Trempe J2")))        Retiré le 18/09/2024

In [0]:
reco_final = f"""
SELECT r.batch, b.production_line, r.target_localization, r.acceptance_status
FROM {source}.recommendations r
JOIN {source}.batches b ON b.id_batch = r.batch AND b.deleted = false
WHERE r.deleted = false
"""

df_reco_final = spark.sql(reco_final)

In [0]:
# Supprimer les recommandations Germination J5 sur ROUEN1
df_reco_final = df_reco_final.filter(~((F.col("production_line") == 1) & (F.col("target_localization") == 14)))

In [0]:
# Créer une fenêtre partitionnée par batch_number et target_localization
windowSpec = Window.partitionBy("batch", "target_localization").orderBy(
    when(col("acceptance_status") == 1, 1)
    .when(col("acceptance_status") == -1, 2)
    .otherwise(3)
)

# Ajouter une colonne de rang en fonction de la priorité sur acceptance_status
df_ranked = df_reco_final.withColumn("rank", row_number().over(windowSpec))

# Garder seulement la première ligne par groupe
df_reco_final = df_ranked.filter(col("rank") == 1).drop("rank")

# GroupBy sur 'batch' et compter les occurrences, puis renommer la colonne 'count' en 'total'
df_reco_final = df_reco_final.groupBy("batch").count().withColumnRenamed("count", "total")

In [0]:
# Supprimer les espaces dans la colonne "acceptance_comments"
df_reco = df_reco.withColumn("acceptance_comments_clean",F.regexp_replace(F.col("acceptance_comments"), "\\s+", ""))

# Ajouter la colonne "applied_status" en utilisant la colonne nettoyée
df_reco = df_reco.withColumn(
    "applied_status",
    F.when(
        (F.col("acceptance_status") == 1) & 
        (F.col("acceptance_comments_clean").rlike("(?i)appli|apli|aplli")) &
        (~F.col("acceptance_comments_clean").rlike("(?i)non|no|pas")),
        "acceptee_et_appliquee"
    ).when(
        F.col("acceptance_status") == 1,
        "acceptee_mais_non_appliquee"
    ).when(
        F.col("acceptance_status") == -1,
        "refusee_ia"
    ).when(
        F.col("acceptance_status") == -2,
        "refusee_usine"
    ).when(
        F.col("acceptance_status") == 0,
        "en_attente"
    ).otherwise(F.lit(None)))

df_reco_save = df_reco.drop("acceptance_comments_clean")

In [0]:
# Sélectionner uniquement les colonnes 'batch', 'target_localization' et 'acceptance_status'
df_reco_batch = df_reco_save.select("batch", "target_localization", "acceptance_status", "applied_status")

# Ajouter la colonne 'status' en fonction de la valeur de 'acceptance_status'
df_reco_batch = df_reco_batch.withColumn(
    "status",
    F.when(F.col("acceptance_status") == -1, "refusee_ia")
     .when(F.col("acceptance_status") == -2, "refusee_usine")
     .when(F.col("acceptance_status") == 0, "en_attente")
     .when(F.col("acceptance_status") == 1, "acceptee"))

# Pivot sur la colonne 'applied_status'
df_reco_batch_applied = df_reco_batch.withColumn("nb_reco", F.lit(1))
df_reco_batch_applied = df_reco_batch_applied.filter(
    (F.col("applied_status") == "acceptee_et_appliquee") |
    (F.col("applied_status") == "acceptee_mais_non_appliquee"))

df_reco_batch_applied = df_reco_batch_applied.groupBy("batch") \
                                            .pivot("applied_status") \
                                            .agg(F.sum("nb_reco"))

# Pivot sur la colonne 'status'
df_reco_batch = df_reco_batch.withColumn("nb_reco", F.lit(1))
df_reco_batch = df_reco_batch.groupBy("batch") \
                              .pivot("status") \
                              .agg(F.sum("nb_reco"))


# À supprimer dès qu'il y aura des valeurs -2
# Vérifie si la colonne existe
if 'refusee_usine' not in df_reco_batch.columns:
    df_reco_batch = df_reco_batch.withColumn('refusee_usine', lit(0))


# Jointure
df_reco_batch = df_reco_batch.join(df_reco_batch_applied.select("batch", "acceptee_et_appliquee", "acceptee_mais_non_appliquee"),
                                   df_reco_batch.batch == df_reco_batch_applied.batch,
                                   how="left") \
                             .drop(df_reco_batch_applied.batch)

# Remplacer les null par des 0
df_reco_batch = df_reco_batch.fillna(0, subset=["acceptee", "en_attente", "refusee_ia", "refusee_usine", "acceptee_et_appliquee", "acceptee_mais_non_appliquee"])

# Ajouter la colonne 'evaluee' avec la somme des colonnes 'acceptee' et 'refusee'
df_reco_batch = df_reco_batch.withColumn("evaluee",F.col("acceptee") + F.col("refusee_ia") + F.col("refusee_usine"))

# Ajouter la colonne 'total_all' avec la somme des colonnes 'acceptee', 'en_attente', et 'refusee'
df_reco_batch = df_reco_batch.withColumn("total_all",F.col("acceptee") + F.col("en_attente") + F.col("refusee_ia") + F.col("refusee_usine"))

# Jointure
df_batches_reco = df_batches.join(df_reco_batch.select("batch", "acceptee", "en_attente", "refusee_ia", "refusee_usine", "acceptee_et_appliquee", "acceptee_mais_non_appliquee", "evaluee", "total_all"),
                                   df_batches.id_batch == df_reco_batch.batch,
                                   how="left").drop(df_reco_batch.batch)

# Jointure pour la colonne 'total'
df_batches_reco = df_batches_reco.join(df_reco_final, df_batches_reco["id_batch"] == df_reco_final["batch"], how="left").drop("batch")

# Jointure pour la colonne 'nb_reco_opti'
df_batches_reco = df_batches_reco.join(df_reco_opti, df_batches_reco["id_batch"] == df_reco_opti["batch"], how="left").drop("batch")

# Remplacer les null par des 0
df_batches_reco = df_batches_reco.fillna(0, subset=["acceptee", "en_attente", "refusee_ia", "refusee_usine", "acceptee_et_appliquee", "acceptee_mais_non_appliquee", "evaluee", "total_all", "total", "nb_reco_opti"])

In [0]:
# Sélectionne les colonnes suivantes
df_batches_reco = df_batches_reco.select("id_batch", "planned_date", "nb_reco_opti", "acceptee", "en_attente", "refusee_ia", "refusee_usine", "acceptee_et_appliquee", "acceptee_mais_non_appliquee", "evaluee", "total_all", "total")

# Trie croissant sur les colonnes suivantes
df_batches_reco = df_batches_reco.orderBy("planned_date")

In [0]:
current_process="fact_batches_reco"

In [0]:
target_fact_batches_reco = current_catalog +"."+current_schema+"."+current_process
print(target_fact_batches_reco)

In [0]:
all_columns =  df_batches_reco.columns
display(all_columns)

In [0]:

# define the primary key 
primary_key = [    
    'id_batch']

additional_columns = get_additional_columns(all_columns, primary_key)

if verbose_mode == 'debug': 
    print(additional_columns)

In [0]:
handle_table_update(
    df_batches_reco, 
    target_fact_batches_reco, 
    primary_key, 
    all_columns,
    additional_columns_to_check=additional_columns,
    mode=execution_mode # Use "update" for update mode, "full" for delete/insert mode
    )